In [1]:
import os
from pathlib import Path
root = Path.cwd()
while not (root / ".git").exists():
    root = root.parent
os.chdir(root)
print(os.getcwd())

import time
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, LeaveOneOut, cross_val_predict
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix

gm = pd.read_csv("results/tables/graph_metrics.csv")
spec = pd.read_csv("results/tables/spectral_features.csv")
chan = pd.read_csv("results/tables/spectral_features_channels.csv")
df = pd.read_csv("data/ds004504/participants.tsv", sep="\t")

PRIMARY = ["strength", "clustering", "path_length", "efficiency", "modularity", "small_world"]
g = gm.pivot(index="participant_id", columns="band", values=PRIMARY)
g.columns = [f"{m}_{b}" for m, b in g.columns]

X_A = chan.drop(columns="Group").set_index("participant_id").join(
    spec.set_index("participant_id")[["theta_alpha_ratio"]])
meta = df.set_index("participant_id")[["Group", "Age"]].loc[X_A.index]
FEATURES = {"A_spectral": X_A, "B_graph": g.loc[X_A.index],
            "C_both": X_A.join(g), "age_only": meta[["Age"]]}
print({k: v.shape for k, v in FEATURES.items()})
print(meta.Group.value_counts().to_dict())

c:\Users\Alive\Documents\eeg-brain-networks-dementia
{'A_spectral': (87, 96), 'B_graph': (87, 30), 'C_both': (87, 126), 'age_only': (87, 1)}
{'A': 36, 'C': 29, 'F': 22}


In [2]:
TASKS = {"AD_vs_CN": (["A", "C"], "A"), "FTD_vs_CN": (["F", "C"], "F"),
         "AD_vs_FTD": (["A", "F"], "A"), "3class": (["A", "F", "C"], None)}

def loso_logistic(X, groups, positive, seed=42):
    ids = meta.index[meta.Group.isin(groups)]
    Xs = X.loc[ids].values
    labels = meta.loc[ids, "Group"].values
    y = (labels == positive).astype(int) if positive else labels
    pipe = Pipeline([("scale", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))])
    search = GridSearchCV(pipe, {"clf__C": [0.01, 0.1, 1, 10]},
                          cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                          scoring="balanced_accuracy")
    proba = cross_val_predict(search, Xs, y, cv=LeaveOneOut(), method="predict_proba", n_jobs=-1)
    out = dict(n=len(ids))
    if positive:
        p1 = proba[:, 1]
        pred = (p1 >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
        out.update(auc=roc_auc_score(y, p1), sensitivity=tp / (tp + fn), specificity=tn / (tn + fp))
    else:
        classes = np.unique(y)
        pred = classes[proba.argmax(axis=1)]
        out.update(auc=roc_auc_score(y, proba, multi_class="ovr", average="macro"),
                   sensitivity=np.nan, specificity=np.nan)
    out["balanced_accuracy"] = balanced_accuracy_score(y, pred)
    preds = pd.DataFrame({"participant_id": ids, "true": y, "pred": pred})
    for j in range(proba.shape[1]):
        preds[f"proba_{j}"] = proba[:, j]
    return out, preds

In [3]:
t = time.time()
res, _ = loso_logistic(FEATURES["A_spectral"], *TASKS["AD_vs_CN"])
print({k: round(v, 3) for k, v in res.items()}, f"{time.time() - t:.0f} s")

{'n': 65, 'auc': 0.863, 'sensitivity': np.float64(0.833), 'specificity': np.float64(0.793), 'balanced_accuracy': 0.813} 31 s


In [4]:
rows, pred_rows = [], []
for task, (groups, pos) in TASKS.items():
    for fname, X in FEATURES.items():
        t = time.time()
        res, preds = loso_logistic(X, groups, pos)
        res.update(task=task, features=fname, model="logistic", seconds=round(time.time() - t))
        rows.append(res)
        preds.insert(0, "task", task); preds.insert(1, "features", fname); preds.insert(2, "model", "logistic")
        pred_rows.append(preds)
        print(task, fname, round(res["balanced_accuracy"], 3), round(res["auc"], 3), f"{res['seconds']} s")
        pd.DataFrame(rows).to_csv("results/tables/classification_results_logistic.csv", index=False)
        pd.concat(pred_rows).to_csv("results/tables/loso_predictions_logistic.csv", index=False)

R = pd.DataFrame(rows)
print(R.pivot(index="features", columns="task", values="balanced_accuracy").round(3))
print(R.pivot(index="features", columns="task", values="auc").round(3))

AD_vs_CN A_spectral 0.813 0.863 15 s
AD_vs_CN B_graph 0.716 0.75 13 s
AD_vs_CN C_both 0.799 0.842 14 s
AD_vs_CN age_only 0.505 0.53 10 s
FTD_vs_CN A_spectral 0.668 0.702 11 s
FTD_vs_CN B_graph 0.543 0.591 10 s
FTD_vs_CN C_both 0.748 0.771 9 s
FTD_vs_CN age_only 0.571 0.6 6 s
AD_vs_FTD A_spectral 0.547 0.515 12 s
AD_vs_FTD B_graph 0.482 0.47 11 s
AD_vs_FTD C_both 0.533 0.484 11 s
AD_vs_FTD age_only 0.551 0.513 7 s
3class A_spectral 0.532 0.696 27 s
3class B_graph 0.373 0.546 18 s
3class C_both 0.473 0.642 22 s
3class age_only 0.343 0.442 11 s
task        3class  AD_vs_CN  AD_vs_FTD  FTD_vs_CN
features                                          
A_spectral   0.532     0.813      0.547      0.668
B_graph      0.373     0.716      0.482      0.543
C_both       0.473     0.799      0.533      0.748
age_only     0.343     0.505      0.551      0.571
task        3class  AD_vs_CN  AD_vs_FTD  FTD_vs_CN
features                                          
A_spectral   0.696     0.863      0.515     

In [5]:
from sklearn.base import clone
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

MODELS = {
    "svm_rbf": (SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42),
                {"clf__C": [0.1, 1, 10]}),
    "random_forest": (RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                              random_state=42, n_jobs=1),
                      {"clf__max_depth": [None, 5]}),
}

def loso(X, groups, positive, model_name, seed=42):
    clf, grid = MODELS[model_name]
    ids = meta.index[meta.Group.isin(groups)]
    Xs = X.loc[ids].values
    labels = meta.loc[ids, "Group"].values
    y = (labels == positive).astype(int) if positive else labels
    pipe = Pipeline([("scale", StandardScaler()), ("clf", clone(clf))])
    search = GridSearchCV(pipe, grid, cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                          scoring="balanced_accuracy")
    proba = cross_val_predict(search, Xs, y, cv=LeaveOneOut(), method="predict_proba", n_jobs=-1)
    out = dict(n=len(ids))
    if positive:
        p1 = proba[:, 1]
        pred = (p1 >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
        out.update(auc=roc_auc_score(y, p1), sensitivity=tp / (tp + fn), specificity=tn / (tn + fp))
    else:
        classes = np.unique(y)
        pred = classes[proba.argmax(axis=1)]
        out.update(auc=roc_auc_score(y, proba, multi_class="ovr", average="macro"),
                   sensitivity=np.nan, specificity=np.nan)
    out["balanced_accuracy"] = balanced_accuracy_score(y, pred)
    preds = pd.DataFrame({"participant_id": ids, "true": y, "pred": pred})
    for j in range(proba.shape[1]):
        preds[f"proba_{j}"] = proba[:, j]
    return out, preds

rows2, pred2 = [], []
for model_name in ["svm_rbf", "random_forest"]:
    for task, (groups, pos) in TASKS.items():
        for fname, X in FEATURES.items():
            t = time.time()
            res, preds = loso(X, groups, pos, model_name)
            res.update(task=task, features=fname, model=model_name, seconds=round(time.time() - t))
            rows2.append(res)
            preds.insert(0, "task", task); preds.insert(1, "features", fname); preds.insert(2, "model", model_name)
            pred2.append(preds)
            print(model_name, task, fname, round(res["balanced_accuracy"], 3), round(res["auc"], 3), f"{res['seconds']} s")
            pd.DataFrame(rows2).to_csv("results/tables/classification_results_other.csv", index=False)
            pd.concat(pred2).to_csv("results/tables/loso_predictions_other.csv", index=False)

R2 = pd.DataFrame(rows2)
for m in ["svm_rbf", "random_forest"]:
    print(m)
    print(R2[R2.model == m].pivot(index="features", columns="task", values="balanced_accuracy").round(3))

full = pd.concat([pd.DataFrame(rows), R2])
full.to_csv("results/tables/classification_results.csv", index=False)

svm_rbf AD_vs_CN A_spectral 0.727 0.784 11 s
svm_rbf AD_vs_CN B_graph 0.482 0.604 8 s
svm_rbf AD_vs_CN C_both 0.765 0.842 13 s
svm_rbf AD_vs_CN age_only 0.444 0.313 15 s
svm_rbf FTD_vs_CN A_spectral 0.516 0.635 20 s
svm_rbf FTD_vs_CN B_graph 0.408 0.423 7 s
svm_rbf FTD_vs_CN C_both 0.709 0.738 9 s
svm_rbf FTD_vs_CN age_only 0.343 0.324 6 s
svm_rbf AD_vs_FTD A_spectral 0.5 0.008 7 s
svm_rbf AD_vs_FTD B_graph 0.486 0.144 14 s
svm_rbf AD_vs_FTD C_both 0.5 0.0 8 s
svm_rbf AD_vs_FTD age_only 0.5 0.0 7 s
svm_rbf 3class A_spectral 0.475 0.582 11 s
svm_rbf 3class B_graph 0.332 0.48 12 s
svm_rbf 3class C_both 0.489 0.629 12 s
svm_rbf 3class age_only 0.327 0.578 16 s
random_forest AD_vs_CN A_spectral 0.716 0.785 215 s
random_forest AD_vs_CN B_graph 0.647 0.724 199 s
random_forest AD_vs_CN C_both 0.734 0.824 194 s
random_forest AD_vs_CN age_only 0.512 0.499 196 s
random_forest FTD_vs_CN A_spectral 0.697 0.668 141 s
random_forest FTD_vs_CN B_graph 0.566 0.552 139 s
random_forest FTD_vs_CN C_both 0

In [6]:
allp = pd.concat([pd.read_csv("results/tables/loso_predictions_logistic.csv"),
                  pd.read_csv("results/tables/loso_predictions_other.csv")])
out = []
rng = np.random.default_rng(0)
for (task, model), d in allp.groupby(["task", "model"]):
    P = d.pivot(index="participant_id", columns="features", values="pred")
    true = d.drop_duplicates("participant_id").set_index("participant_id")["true"].loc[P.index].values
    n = len(true)
    boots = {f: [] for f in P.columns}
    diffs = []
    for _ in range(2000):
        idx = rng.integers(0, n, n)
        t = true[idx]
        if len(np.unique(t)) < len(np.unique(true)):
            continue
        ba = {f: balanced_accuracy_score(t, P[f].values[idx]) for f in P.columns}
        for f in ba:
            boots[f].append(ba[f])
        diffs.append(ba["C_both"] - ba["A_spectral"])
    for f in P.columns:
        out.append(dict(task=task, model=model, features=f, balanced_accuracy=balanced_accuracy_score(true, P[f]),
                        ci_low=np.percentile(boots[f], 2.5), ci_high=np.percentile(boots[f], 97.5)))
    obs = balanced_accuracy_score(true, P["C_both"]) - balanced_accuracy_score(true, P["A_spectral"])
    out.append(dict(task=task, model=model, features="C minus A", balanced_accuracy=obs,
                    ci_low=np.percentile(diffs, 2.5), ci_high=np.percentile(diffs, 97.5)))
ci = pd.DataFrame(out)
ci.to_csv("results/tables/classification_intervals.csv", index=False)
print(ci[ci.features == "C minus A"].round(3).to_string(index=False))
print(ci[ci.features != "C minus A"].round(3).to_string(index=False))

     task         model  features  balanced_accuracy  ci_low  ci_high
   3class      logistic C minus A             -0.059  -0.147    0.025
   3class random_forest C minus A             -0.012  -0.090    0.067
   3class       svm_rbf C minus A              0.014  -0.038    0.066
 AD_vs_CN      logistic C minus A             -0.014  -0.069    0.038
 AD_vs_CN random_forest C minus A              0.017  -0.060    0.103
 AD_vs_CN       svm_rbf C minus A              0.038  -0.057    0.130
AD_vs_FTD      logistic C minus A             -0.014  -0.161    0.131
AD_vs_FTD random_forest C minus A              0.019  -0.085    0.111
AD_vs_FTD       svm_rbf C minus A              0.000   0.000    0.000
FTD_vs_CN      logistic C minus A              0.080  -0.009    0.179
FTD_vs_CN random_forest C minus A              0.005  -0.048    0.067
FTD_vs_CN       svm_rbf C minus A              0.194   0.071    0.317
     task         model   features  balanced_accuracy  ci_low  ci_high
   3class      logi